In [1]:
print("hello")

hello


In [5]:
import cv2
from moviepy.editor import *
import os

def zoom_in(image, duration, fps, size):
    frames = []
    h, w = image.shape[:2]
    for i in range(int(duration * fps)):
        scale = 1 + (i / (duration * fps) * 0.5)
        center = (w // 2, h // 2)
        matrix = cv2.getRotationMatrix2D(center, 0, scale)
        frame = cv2.warpAffine(image, matrix, (w, h))
        frames.append(cv2.resize(frame, size))
    return frames

def zoom_out(image, duration, fps, size):
    frames = []
    h, w = image.shape[:2]
    for i in range(int(duration * fps)):
        scale = 1 + ((duration * fps - i) / (duration * fps) * 0.5)
        center = (w // 2, h // 2)
        matrix = cv2.getRotationMatrix2D(center, 0, scale)
        frame = cv2.warpAffine(image, matrix, (w, h))
        frames.append(cv2.resize(frame, size))
    return frames

def create_video(image_files, display_durations, size=(1080, 1920), fps=30):
    images = [cv2.resize(cv2.imread(img), size) for img in image_files]
    clips = []
    for i in range(len(images)):
        img = images[i]
        display_duration = display_durations[i]
        if i % 2 == 0:
            zoom_frames = zoom_in(img, display_duration, fps, size)
        else:
            zoom_frames = zoom_out(img, display_duration, fps, size)

        # Convert frames to video clips
        frames_clips = [ImageSequenceClip([cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)], fps=fps) for frame in zoom_frames]
        clip = concatenate_videoclips(frames_clips)
        clips.append(clip)

    # Combine all video clips into one video
    video_clip = concatenate_videoclips(clips, method="compose")
    return video_clip

def load_images_and_voices(images_root_folder, voices_root_folder):
    # Load images from the given folder
    images = sorted([os.path.join(images_root_folder, img) for img in os.listdir(images_root_folder) if img.endswith('.png')])
    
    # Load voices from the given folder
    voices = sorted([os.path.join(voices_root_folder, voice) for voice in os.listdir(voices_root_folder) if voice.endswith('.mp3')])
    
    return images, voices

def main():
    images_root_folder = 'Generated_images/images_20240801092231'
    voices_root_folder = 'Generated_voice/voice_20240801100130'
    images, voices = load_images_and_voices(images_root_folder, voices_root_folder)
    
    # Calculate display durations from audio files
    display_durations = []
    for file_path in voices:
        try:
            audio = AudioFileClip(file_path)
            duration_sec = audio.duration
            display_durations.append(duration_sec)
        except Exception as e:
            print(f"Error processing {file_path}: {e}")
    
    video_clip = create_video(images, display_durations)
    
    # Combine audio files into one
    audio_clips = [AudioFileClip(path) for path in voices]
    combined_audio = concatenate_audioclips(audio_clips)
    
    if combined_audio.duration > video_clip.duration:
        combined_audio = combined_audio.subclip(0, video_clip.duration)

    # Set audio to video
    final_clip = video_clip.set_audio(combined_audio)
    final_clip.write_videofile('final_video.mp4')

    print("Combined video file created: final_video.mp4")

if __name__ == "__main__":
    main()


Moviepy - Building video final_video.mp4.
MoviePy - Writing audio in final_videoTEMP_MPY_wvf_snd.mp3


MoviePy - Done.
Moviepy - Writing video final_video.mp4



Moviepy - Done !
Moviepy - video ready final_video.mp4
Combined video file created: final_video.mp4
